### Final Test PCR

In [16]:
import joblib
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, MetaEstimatorMixin, TransformerMixin

class ForcedInclusionSelector(BaseEstimator, MetaEstimatorMixin, TransformerMixin):
    """
    A feature selector that forces certain features to be included
    regardless of the selection made by the underlying selector.

    Parameters:
    selector: An instance of a feature selector (e.g., SelectKBest).
    vital_indices: List or array of feature indices that must be included.

    Methods:
    fit(X, y): Fits the underlying selector to the data
    transform(X): Transforms the data to include selected features and vital features
    get_support(indices=False): Returns a mask or indices of the selected features
    """
    def __init__(self, selector, vital_indices):
        self.selector = selector
        self.vital_indices = vital_indices

    def fit(self, X, y=None):
        # Fit the underlying selector
        self.selector.fit(X, y)
        return self

    def transform(self, X):
        # Get the support mask from the underlying selector
        mask = self.selector.get_support().copy()

        # Ensure vital features are included
        if self.vital_indices:
            mask[self.vital_indices] = True
        return X[:, mask]

    def get_support(self, indices=False):
        # Get the support mask from the underlying selector
        mask = self.selector.get_support().copy()
        # Ensure vital features are included
        if self.vital_indices:
            mask[self.vital_indices] = True
        return mask if not indices else np.where(mask)[0]

In [17]:
OUTPUT_FILE_PATH = "../results/PCRPrediction.csv"

# Load the model from the file
final_model = joblib.load('models/pcr_final_model.pkl')

In [18]:
# Sanity check on example test dataset
test_example = pd.read_csv("../data/TestDatasetExample.csv")
test_example_clean = test_example.replace(999, np.nan)
y_pred_example = final_model.predict(test_example_clean)
pd.Series(y_pred_example).value_counts()

1.0    2
0.0    1
Name: count, dtype: int64

In [19]:
# Predict on the actual test dataset
TEST_DATASET_PATH = "../data/FinalTestDataset2025.csv"
test = pd.read_csv(TEST_DATASET_PATH)

test.shape

(133, 119)

In [20]:
# Clean the test data
test_clean = test.replace(999, np.nan)

# Make predictions
y_pred = final_model.predict(test_clean)
pd.Series(y_pred).value_counts()

1.0    67
0.0    66
Name: count, dtype: int64

In [21]:
# Save the predictions to a CSV file
output_df = pd.DataFrame()
output_df['ID'] = test['ID']
output_df['PCR_Prediction'] = y_pred

output_df.to_csv(OUTPUT_FILE_PATH, index=False)
print(f"\n[SUCCESS] Predictions saved to '{OUTPUT_FILE_PATH}'")


[SUCCESS] Predictions saved to '../results/PCRPrediction.csv'
